# CSE 234 Programming Assignment 3: Speculative Decoding

## Setup

In [2]:
import os
import torch
import time
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import List, Tuple, Dict, Optional

## Speculative Decoding

In [17]:
import time
import torch

from typing import Tuple, Dict
from transformers import AutoTokenizer, AutoModelForCausalLM


class SpeculativeDecoder:
    def __init__(
        self,
        target_model_name: str,
        draft_model_name: str,
        device: str = "cuda",
    ):
        self.device = device

        self.target_model, self.target_tokenizer = self.initialize_target_model(
            target_model_name
        )
        self.draft_model, self.draft_tokenizer = self.initialize_draft_model(
            draft_model_name
        )

        # Pythia-1.4B and Pythia-160M should use the same tokenizer/vocab.
        assert (
            self.target_tokenizer.get_vocab()
            == self.draft_tokenizer.get_vocab()
        ), "Target and draft tokenizers must be compatible."

        self.last_generated_token_count = 0

    # ============================================================
    # Model initialization
    # ============================================================

    def initialize_target_model(self, model_name: str):
        print(f"Loading target model: {model_name}")

        tokenizer = AutoTokenizer.from_pretrained(model_name)

        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token

        dtype = (
            torch.float16
            if torch.device(self.device).type == "cuda"
            else torch.float32
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
        )

        model = model.to(self.device)
        model.eval()

        # Enable KV cache for generation.
        model.config.use_cache = True

        # These fields expect token IDs, not token strings.
        model.config.pad_token_id = tokenizer.pad_token_id
        model.generation_config.pad_token_id = tokenizer.pad_token_id

        return model, tokenizer

    def initialize_draft_model(self, model_name: str):
        print(f"Loading draft model: {model_name}")

        tokenizer = AutoTokenizer.from_pretrained(model_name)

        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token

        dtype = (
            torch.float16
            if torch.device(self.device).type == "cuda"
            else torch.float32
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
        )

        model = model.to(self.device)
        model.eval()

        model.config.use_cache = True
        model.config.pad_token_id = tokenizer.pad_token_id
        model.generation_config.pad_token_id = tokenizer.pad_token_id

        return model, tokenizer

    # ============================================================
    # Draft generation
    # ============================================================

    def generate_draft_tokens(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        num_speculative_tokens: int = 10,
    ) -> torch.Tensor:

        with torch.inference_mode():
            output_ids = self.draft_model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=num_speculative_tokens,
                do_sample=False,
                use_cache=True,
                pad_token_id=self.draft_tokenizer.pad_token_id,
                eos_token_id=self.draft_tokenizer.eos_token_id,
            )

        prompt_length = input_ids.shape[1]

        # [B, L + k] -> [B, k]
        draft_tokens = output_ids[:, prompt_length:]

        return draft_tokens

    # ============================================================
    # Vectorized verification
    # ============================================================

    def verify_tokens_vectorized(
        self,
        input_ids: torch.Tensor,
        draft_tokens: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> Tuple[torch.Tensor, int, torch.Tensor]:
        """
        Verify all draft tokens with a single target-model forward pass.

        Returns:
            accepted_tokens:
                Shape [1, accepted_position]

            accepted_position:
                Number of consecutively accepted draft tokens.

            next_token:
                If a mismatch occurs:
                    target model's token at the mismatch position.

                If all draft tokens are accepted:
                    bonus token predicted after the final draft token.
        """

        # This lab implementation assumes batch size = 1.
        assert input_ids.shape[0] == 1

        draft_length = draft_tokens.shape[1]

        if draft_length == 0:
            raise ValueError("draft_tokens must contain at least one token.")

        prompt_length = input_ids.shape[1]

        # --------------------------------------------------------
        # [prompt] + [draft tokens]
        # --------------------------------------------------------

        concatenated_ids = torch.cat(
            [input_ids, draft_tokens],
            dim=1,
        )

        # Original attention mask only covers the prompt.
        # Extend it for all draft tokens.
        verification_attention_mask = torch.cat(
            [
                attention_mask,
                torch.ones(
                    (attention_mask.shape[0], draft_length),
                    dtype=attention_mask.dtype,
                    device=attention_mask.device,
                ),
            ],
            dim=1,
        )

        # --------------------------------------------------------
        # One target-model forward
        # --------------------------------------------------------

        with torch.inference_mode():
            outputs = self.target_model(
                input_ids=concatenated_ids,
                attention_mask=verification_attention_mask,
                use_cache=False,
            )

        logits = outputs.logits
        # shape:
        # [1, prompt_length + draft_length, vocab_size]

        # --------------------------------------------------------
        # Which logits predict the draft tokens?
        #
        # logits[L - 1] -> draft[0]
        # logits[L]     -> draft[1]
        # ...
        # logits[L+k-2] -> draft[k-1]
        # --------------------------------------------------------

        verification_logits = logits[
            :,
            prompt_length - 1 : prompt_length + draft_length - 1,
            :,
        ]

        # [1, k]
        target_predictions = verification_logits.argmax(dim=-1)

        # --------------------------------------------------------
        # Find first mismatch
        # --------------------------------------------------------

        matches = (
            target_predictions[0]
            == draft_tokens[0]
        )

        mismatch_positions = torch.nonzero(
            ~matches,
            as_tuple=False,
        )

        if mismatch_positions.numel() > 0:
            # First rejected draft token.
            accepted_position = mismatch_positions[0].item()

            accepted_tokens = draft_tokens[
                :,
                :accepted_position,
            ]

            # Target's prediction at the rejected position.
            next_token = target_predictions[
                :,
                accepted_position : accepted_position + 1,
            ]

        else:
            # All k draft tokens accepted.
            accepted_position = draft_length
            accepted_tokens = draft_tokens

            # ----------------------------------------------------
            # Bonus token
            #
            # logits[L+k-1] predicts the token AFTER draft[k-1].
            # ----------------------------------------------------

            bonus_logits = logits[
                :,
                prompt_length + draft_length - 1,
                :,
            ]

            next_token = bonus_logits.argmax(
                dim=-1,
                keepdim=True,
            )

        return accepted_tokens, accepted_position, next_token

    # ============================================================
    # Main speculative decoding
    # ============================================================

    def speculative_decode(
        self,
        prompt: str,
        max_tokens: int = 100,
        num_speculative_tokens: int = 15,
    ) -> str:

        inputs = self.target_tokenizer(
            prompt,
            return_tensors="pt",
            padding=True,
        )

        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)

        prompt_length = input_ids.shape[1]

        total_draft_tokens_proposed = 0
        total_draft_tokens_accepted = 0
        generated_count = 0

        if torch.device(self.device).type == "cuda":
            torch.cuda.synchronize()

        start_time = time.time()

        while generated_count < max_tokens:

            remaining = max_tokens - generated_count

            # ----------------------------------------------------
            # If only one token remains, speculative decoding
            # cannot give us any benefit. Just use target model.
            # ----------------------------------------------------

            if remaining == 1:

                with torch.inference_mode():
                    outputs = self.target_model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        use_cache=False,
                    )

                next_token = outputs.logits[
                    :,
                    -1,
                    :,
                ].argmax(
                    dim=-1,
                    keepdim=True,
                )

                new_tokens = next_token

            else:
                # We reserve one position for correction / bonus token.
                draft_k = min(
                    num_speculative_tokens,
                    remaining - 1,
                )

                # ------------------------------------------------
                # 1. Draft proposes k tokens
                # ------------------------------------------------

                draft_tokens = self.generate_draft_tokens(
                    input_ids,
                    attention_mask,
                    draft_k,
                )

                if draft_tokens.shape[1] == 0:
                    break

                total_draft_tokens_proposed += draft_tokens.shape[1]

                # ------------------------------------------------
                # 2. Target verifies all k tokens simultaneously
                # ------------------------------------------------

                (
                    accepted_tokens,
                    accepted_position,
                    next_token,
                ) = self.verify_tokens_vectorized(
                    input_ids,
                    draft_tokens,
                    attention_mask,
                )

                total_draft_tokens_accepted += accepted_position

                # ------------------------------------------------
                # 3. Commit:
                #
                # accepted draft tokens
                # +
                # correction / bonus token
                # ------------------------------------------------

                new_tokens = torch.cat(
                    [
                        accepted_tokens,
                        next_token,
                    ],
                    dim=1,
                )

            # Safety: never exceed max_tokens.
            new_tokens = new_tokens[:, :remaining]

            # ----------------------------------------------------
            # EOS handling
            # ----------------------------------------------------

            should_stop = False
            eos_token_id = self.target_tokenizer.eos_token_id

            if eos_token_id is not None:

                eos_positions = torch.nonzero(
                    new_tokens[0] == eos_token_id,
                    as_tuple=False,
                )

                if eos_positions.numel() > 0:
                    first_eos = eos_positions[0].item()

                    # Keep EOS itself, remove everything after it.
                    new_tokens = new_tokens[
                        :,
                        : first_eos + 1,
                    ]

                    should_stop = True

            # ----------------------------------------------------
            # Update sequence
            # ----------------------------------------------------

            input_ids = torch.cat(
                [input_ids, new_tokens],
                dim=1,
            )

            attention_mask = torch.cat(
                [
                    attention_mask,
                    torch.ones(
                        (
                            attention_mask.shape[0],
                            new_tokens.shape[1],
                        ),
                        dtype=attention_mask.dtype,
                        device=attention_mask.device,
                    ),
                ],
                dim=1,
            )

            generated_count += new_tokens.shape[1]

            if should_stop:
                break

        if torch.device(self.device).type == "cuda":
            torch.cuda.synchronize()

        elapsed_time = time.time() - start_time

        self.last_generated_token_count = generated_count

        acceptance_rate = (
            total_draft_tokens_accepted
            / total_draft_tokens_proposed
            if total_draft_tokens_proposed > 0
            else 0.0
        )

        print(
            f"Generated {generated_count} tokens "
            f"in {elapsed_time:.2f} seconds"
        )

        print(
            f"Tokens per second: "
            f"{generated_count / elapsed_time:.2f}"
        )

        print(
            f"Draft token acceptance rate: "
            f"{acceptance_rate:.2%}"
        )

        return self.target_tokenizer.decode(
            input_ids[0],
            skip_special_tokens=True,
        )

    # ============================================================
    # Benchmark
    # ============================================================

    def benchmark(
        self,
        prompt: str,
        max_tokens: int = 100,
        num_runs: int = 3,
        compare_baseline: bool = True,
    ) -> Dict:

        results = {
            "speculative": {
                "times": [],
                "tokens_per_second": [],
            },
            "baseline": (
                {
                    "times": [],
                    "tokens_per_second": [],
                }
                if compare_baseline
                else None
            ),
        }

        # --------------------------------------------------------
        # Speculative decoding
        # --------------------------------------------------------

        for _ in range(num_runs):

            if torch.device(self.device).type == "cuda":
                torch.cuda.synchronize()

            start_time = time.time()

            self.speculative_decode(
                prompt,
                max_tokens=max_tokens,
            )

            if torch.device(self.device).type == "cuda":
                torch.cuda.synchronize()

            elapsed = time.time() - start_time

            output_tokens = self.last_generated_token_count

            tps = (
                output_tokens / elapsed
                if elapsed > 0
                else 0.0
            )

            results["speculative"]["times"].append(elapsed)
            results["speculative"]["tokens_per_second"].append(tps)

        # --------------------------------------------------------
        # Baseline greedy decoding
        # --------------------------------------------------------

        if compare_baseline:

            for _ in range(num_runs):

                inputs = self.target_tokenizer(
                    prompt,
                    return_tensors="pt",
                    padding=True,
                )

                input_ids = inputs["input_ids"].to(self.device)
                attention_mask = inputs["attention_mask"].to(self.device)

                if torch.device(self.device).type == "cuda":
                    torch.cuda.synchronize()

                start_time = time.time()

                with torch.inference_mode():
                    output_ids = self.target_model.generate(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        max_new_tokens=max_tokens,
                        do_sample=False,
                        use_cache=True,
                        pad_token_id=self.target_tokenizer.pad_token_id,
                        eos_token_id=self.target_tokenizer.eos_token_id,
                    )

                if torch.device(self.device).type == "cuda":
                    torch.cuda.synchronize()

                elapsed = time.time() - start_time

                output_tokens = (
                    output_ids.shape[1]
                    - input_ids.shape[1]
                )

                tps = (
                    output_tokens / elapsed
                    if elapsed > 0
                    else 0.0
                )

                results["baseline"]["times"].append(elapsed)
                results["baseline"]["tokens_per_second"].append(tps)

        # --------------------------------------------------------
        # Average metrics
        # --------------------------------------------------------

        for method in results:
            if results[method] is not None:

                results[method]["avg_time"] = (
                    sum(results[method]["times"])
                    / num_runs
                )

                results[method]["avg_tokens_per_second"] = (
                    sum(
                        results[method][
                            "tokens_per_second"
                        ]
                    )
                    / num_runs
                )

        if compare_baseline:

            baseline_time = results["baseline"]["avg_time"]
            speculative_time = results["speculative"]["avg_time"]

            results["speedup"] = (
                baseline_time
                / speculative_time
            )

            results["latency_reduction"] = (
                1
                - speculative_time
                / baseline_time
            ) * 100

        return results

## Test

In [18]:
target_model_name = "EleutherAI/pythia-1.4b-deduped"  # Larger target model
draft_model_name = "EleutherAI/pythia-160m-deduped"   # Smaller draft model


# Initialize speculative decoder
decoder = SpeculativeDecoder(
    target_model_name=target_model_name,
    draft_model_name=draft_model_name,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# Test prompts
test_prompts = [
    "The future of Artificial Intelligence is",
    "Write a short story about a robot learning to feel emotions:",
    "Write the lyrics to the song 'Happy Birthday'."
]

# Run benchmark on test prompts
for i, prompt in enumerate(test_prompts):
    print(f"\nBenchmarking Prompt {i+1}:")
    print(f"Prompt: {prompt}")

    results = decoder.benchmark(
        prompt=prompt,
        max_tokens=100,
        num_runs=3,
        compare_baseline=True
    )

    print(f"Average speculative decoding time: {results['speculative']['avg_time']:.2f} seconds")
    print(f"Average speculative tokens per second: {results['speculative']['avg_tokens_per_second']:.2f}")

    if results["baseline"] is not None:
        print(f"Average baseline decoding time: {results['baseline']['avg_time']:.2f} seconds")
        print(f"Average baseline tokens per second: {results['baseline']['avg_tokens_per_second']:.2f}")
        print(f"Speedup: {results['speedup']:.2f}x")
        print(f"Latency reduction: {results['latency_reduction']:.2f}%")

Loading target model: EleutherAI/pythia-1.4b-deduped


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading draft model: EleutherAI/pythia-160m-deduped


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Benchmarking Prompt 1:
Prompt: The future of Artificial Intelligence is
Generated 100 tokens in 1.35 seconds
Tokens per second: 74.24
Draft token acceptance rate: 85.98%
Generated 100 tokens in 1.16 seconds
Tokens per second: 86.57
Draft token acceptance rate: 85.98%
Generated 100 tokens in 1.11 seconds
Tokens per second: 89.86
Draft token acceptance rate: 85.98%
Average speculative decoding time: 1.22 seconds
Average speculative tokens per second: 82.90
Average baseline decoding time: 1.77 seconds
Average baseline tokens per second: 56.55
Speedup: 1.46x
Latency reduction: 31.37%

Benchmarking Prompt 2:
Prompt: Write a short story about a robot learning to feel emotions:
Generated 100 tokens in 1.37 seconds
Tokens per second: 72.98
Draft token acceptance rate: 93.00%
Generated 100 tokens in 1.51 seconds
Tokens per second: 66.02
Draft token acceptance rate: 93.00%
Generated 100 tokens in 1.06 seconds
Tokens per second: 94.67
Draft token acceptance rate: 93.00%
Average speculative decod

## Bonus

In [19]:
target_model_name = ...  # Larger target model
draft_model_name = ...   # Smaller draft model


# Initialize speculative decoder
decoder = SpeculativeDecoder(
    target_model_name=target_model_name,
    draft_model_name=draft_model_name,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# Test prompts
test_prompts = [
    "The future of Artificial Intelligence is",
    "Write a short story about a robot learning to feel emotions:",
    "Write the lyrics to the song 'Happy Birthday'."
]

# Run benchmark on test prompts
for i, prompt in enumerate(test_prompts):
    print(f"\nBenchmarking Prompt {i+1}:")
    print(f"Prompt: {prompt}")

    results = decoder.benchmark(
        prompt=prompt,
        max_tokens=100,
        num_runs=3,
        compare_baseline=True
    )

    print(f"Average speculative decoding time: {results['speculative']['avg_time']:.2f} seconds")
    print(f"Average speculative tokens per second: {results['speculative']['avg_tokens_per_second']:.2f}")

    if results["baseline"] is not None:
        print(f"Average baseline decoding time: {results['baseline']['avg_time']:.2f} seconds")
        print(f"Average baseline tokens per second: {results['baseline']['avg_tokens_per_second']:.2f}")
        print(f"Speedup: {results['speedup']:.2f}x")
        print(f"Latency reduction: {results['latency_reduction']:.2f}%")

Loading target model: Ellipsis


OSError: Ellipsis is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`